# Q2.5 Dead Neurons

In [1]:
import sys
from pathlib import Path
import argparse
import numpy as np
import wandb
from sklearn.metrics import precision_recall_fscore_support

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.data_loader import load_dataset
from src.ann.neural_network import NeuralNetwork

ENTITY = 'anandhakrishnanm21-indian-institute-of-technology-madras'


In [5]:
def to_namespace(cfg):
    d = dict(cfg)
    defaults = {
        'hidden_size': [128, 128],
        'activation': 'relu',
        'weight_init': 'xavier',
        'loss': 'cross_entropy',
        'learning_rate': 1e-3,
        'weight_decay': 0.0,
        'optimizer': 'sgd',
        'epochs': 10,
        'batch_size': 64,
    }
    defaults.update(d)
    defaults['num_layers'] = len(defaults['hidden_size'])
    return argparse.Namespace(**defaults)


def evaluate_split(model, X, Y):
    logits = model.forward(X)
    loss, acc = model.evaluate(X, Y)
    y_true = np.argmax(Y, axis=1)
    y_pred = np.argmax(logits, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    return {
        'loss': float(loss),
        'acc': float(acc),
        'precision': float(p),
        'recall': float(r),
        'f1': float(f1),
        'y_true': y_true,
        'y_pred': y_pred,
    }


def train_one_run(cfg, project, run_name, dataset_name='mnist', extras_fn=None):
    run = wandb.init(project=project, name=run_name, config=cfg, reinit=True)
    args = to_namespace(wandb.config)

    Xtr, Ytr, Xva, Yva, Xte, Yte = load_dataset(dataset_name)
    model = NeuralNetwork(args)

    best_val_f1 = -1.0
    for epoch in range(args.epochs):
        model.train(Xtr, Ytr, epochs=1, batch_size=args.batch_size, X_val=Xva, y_val=Yva, verbose=False)
        tr = evaluate_split(model, Xtr, Ytr)
        va = evaluate_split(model, Xva, Yva)
        te = evaluate_split(model, Xte, Yte)

        payload = {
            'epoch': epoch + 1,
            'train_loss': tr['loss'], 'train_acc': tr['acc'], 'train_f1': tr['f1'],
            'val_loss': va['loss'], 'val_acc': va['acc'], 'val_f1': va['f1'],
            'test_loss': te['loss'], 'test_acc': te['acc'], 'test_f1': te['f1'],
            'test_precision': te['precision'], 'test_recall': te['recall'],
        }
        if extras_fn is not None:
            payload.update(extras_fn(model, Xva, Yva))
        wandb.log(payload)

        if va['f1'] > best_val_f1:
            best_val_f1 = va['f1']

    wandb.summary['best_val_f1'] = float(best_val_f1)
    wandb.finish()


In [6]:
def dead_neuron_extras(model, X_val, Y_val):
    probe_n = min(1024, X_val.shape[0])
    X_probe = X_val[:probe_n]
    Y_probe = Y_val[:probe_n]

    logits = model.forward(X_probe)
    _ = model.backward(Y_probe, logits)

    out = {}
    for i in range(1, len(model._a_cache)):
        a = model._a_cache[i]
        out[f'dead_ratio_layer_{i}'] = float(np.mean(np.all(np.isclose(a, 0.0), axis=0)))
        out[f'activation_mean_layer_{i}'] = float(np.mean(a))
        out[f'activation_std_layer_{i}'] = float(np.std(a))
        out[f'activation_hist_layer_{i}'] = wandb.Histogram(a.reshape(-1))

    for i, layer in enumerate(model.layers, start=1):
        gw_norm = float(np.linalg.norm(layer.grad_W))
        gb_norm = float(np.linalg.norm(layer.grad_b))
        out[f'grad_norm_W_layer_{i}'] = gw_norm
        out[f'grad_norm_b_layer_{i}'] = gb_norm
        out[f'grad_norm_layer_{i}'] = float(np.sqrt(gw_norm * gw_norm + gb_norm * gb_norm))

    return out

PROJECT = 'q2_5_dead_neurons'
for activation in ['relu', 'tanh']:
    cfg = {
        'epochs': 20, 'batch_size': 64, 'learning_rate': 0.1,
        'weight_decay': 0.0, 'optimizer': 'rmsprop', 'activation': activation,
        'hidden_size': [128, 128, 128], 'weight_init': 'xavier', 'loss': 'cross_entropy'
    }
    train_one_run(cfg, project=PROJECT, run_name=f'dead_{activation}_lr0.1', dataset_name='mnist', extras_fn=dead_neuron_extras)



activation_mean_layer_1,▁▁▄██▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
activation_mean_layer_2,█▆▄▃▅▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
activation_mean_layer_3,▄█▃▃▄▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
activation_std_layer_1,▁▂▄██▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
activation_std_layer_2,█▆▅▄▅▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
activation_std_layer_3,▅█▄▄▄▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
dead_ratio_layer_1,▁███████████████████
dead_ratio_layer_2,▁▅▅▅▅███████████████
dead_ratio_layer_3,▆▁▆▆▆███████████████
epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
+23,...


activation_mean_layer_1,███▇▇▇▇▇▇▇▇▁▁▁▁▁▁▁▁▁
activation_mean_layer_2,▁▁▁▁▁▁▁▁▃▃▄█████████
activation_mean_layer_3,████████▁▁▁▁▁▁▁▁▁▁▁▁
activation_std_layer_1,▁▁▁▁▁▁▁▁▂▂▂█████████
activation_std_layer_2,▁▁▁▂▂▂▂▂▃▃▄█████████
activation_std_layer_3,▁▁▁▁▁▁▁▁████████████
dead_ratio_layer_1,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
dead_ratio_layer_2,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
dead_ratio_layer_3,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
+23,...
